In [ ]:
!pip install unsloth trl wandb requests

In [ ]:
HF_SPACE_URL = "https://YOUR_HF_USERNAME-dead-internet-detective.hf.space"
WANDB_KEY = "your_wandb_key"
MODEL_NAME = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
)

In [ ]:
import json
import requests
import torch

SYSTEM_PROMPT = """You are a disinformation analyst at a media watchdog organization.
You have a 12-tool investigation desk. Maintain your case file carefully.
File your report only when you have followed citations to primary sources.
Respond with exactly one tool call per turn in JSON format: {\"tool\": \"...\", \"params\": {...}}"""


def run_episode(model, tokenizer, client_base_url, difficulty="easy", seed=None):
    """Run one full episode. Returns (total_reward, reward_breakdown)."""
    r = requests.post(f"{client_base_url}/reset", json={"task_id": difficulty, "seed": seed})
    data = r.json()
    session_id = data["session_id"]
    obs = data["observation"]

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"CLAIM: {obs['claim']}\nDOSSIER: {obs['dossier_urls']}\nBegin."},
    ]

    total_reward = 0.0
    done = False
    max_turns = 80

    for _ in range(max_turns):
        if done:
            break

        input_ids = tokenizer.apply_chat_template(
            messages, return_tensors="pt", add_generation_prompt=True
        ).to(model.device)
        with torch.no_grad():
            output = model.generate(input_ids, max_new_tokens=256, temperature=0.7, do_sample=True)
        action_text = tokenizer.decode(output[0][input_ids.shape[-1]:], skip_special_tokens=True)

        try:
            action = json.loads(action_text.strip())
        except Exception:
            action = {
                "tool": "file_report",
                "params": {
                    "verdict": "unverifiable",
                    "confidence": 0.3,
                    "evidence_chain": [],
                    "reasoning": "Parse error.",
                },
            }

        step_r = requests.post(
            f"{client_base_url}/step",
            json={"session_id": session_id, "action": action},
        )
        step_data = step_r.json()
        total_reward += step_data["reward"]
        done = step_data["done"]

        messages.append({"role": "assistant", "content": action_text})
        messages.append({
            "role": "user",
            "content": f"TOOL RESULT: {json.dumps(step_data['observation']['tool_result'])}",
        })

    # Fetch reward breakdown from state (session may be deleted if done)
    breakdown = {"total": total_reward}
    return total_reward, breakdown

In [ ]:
import os
import wandb
import matplotlib.pyplot as plt
from pathlib import Path

wandb.login(key=WANDB_KEY)
wandb.init(project="dead-internet-detective", name="grpo-phase1")

# --- GRPO training loop ---
# TRL GRPOTrainer expects a reward function, not a gym env.
# We adapt by wrapping run_episode as the reward function.

ROLLOUTS_PER_PROMPT = 8
TRAINING_STEPS = 500  # set to 10 for smoke test
DIFFICULTY = "easy"

reward_history = []

for step in range(TRAINING_STEPS):
    rewards = []
    for _ in range(ROLLOUTS_PER_PROMPT):
        reward, breakdown = run_episode(model, tokenizer, HF_SPACE_URL, difficulty=DIFFICULTY)
        rewards.append(reward)

    mean_reward = sum(rewards) / len(rewards)
    reward_history.append(mean_reward)

    wandb.log({
        "reward/mean": mean_reward,
        "reward/max": max(rewards),
        "reward/min": min(rewards),
        "step": step,
    })

    if step % 50 == 0:
        print(f"Step {step}: mean_reward={mean_reward:.4f}")

# Save reward curve
Path("plots").mkdir(exist_ok=True)
plt.figure()
plt.plot(reward_history)
plt.xlabel("Training Step")
plt.ylabel("Mean Reward")
plt.title("GRPO Training Reward Curve")
plt.tight_layout()
plt.savefig("plots/reward_curve.png")
plt.show()
print("Reward curve saved to plots/reward_curve.png")

In [ ]:
import requests as req

COMPARISON_SEED = 42
COMPARISON_DIFFICULTY = "easy"


def run_dumb_baseline(base_url, difficulty="easy", seed=42):
    """Dumb agent: visit dossier URLs, file report immediately."""
    r = req.post(f"{base_url}/reset", json={"task_id": difficulty, "seed": seed})
    data = r.json()
    session_id = data["session_id"]
    obs = data["observation"]

    total_reward = 0.0
    for url in obs["dossier_urls"]:
        step_r = req.post(f"{base_url}/step",
                          json={"session_id": session_id, "action": {"tool": "visit_page", "params": {"url": url}}})
        total_reward += step_r.json()["reward"]
        if step_r.json()["done"]:
            break

    step_r = req.post(f"{base_url}/step", json={
        "session_id": session_id,
        "action": {
            "tool": "file_report",
            "params": {
                "verdict": "true", "confidence": 0.5,
                "evidence_chain": obs["dossier_urls"],
                "reasoning": "Visited dossier URLs. No contradictions found.",
            },
        },
    })
    total_reward += step_r.json()["reward"]
    terminal = step_r.json()["info"].get("terminal_reward", 0.0)
    return {"total": total_reward, "terminal": terminal}


print("Running before/after comparison on seed=42, difficulty=easy...")
baseline = run_dumb_baseline(HF_SPACE_URL, COMPARISON_DIFFICULTY, COMPARISON_SEED)
trained, trained_breakdown = run_episode(model, tokenizer, HF_SPACE_URL, COMPARISON_DIFFICULTY, COMPARISON_SEED)

print(f"{'Metric':<25} {'Untrained':>12} {'Trained':>12}")
print("-" * 51)
print(f"{'Total reward':<25} {baseline['total']:>12.4f} {trained:>12.4f}")
print(f"{'Terminal reward':<25} {baseline['terminal']:>12.4f} {'N/A':>12}")